# Fine-Tune DeBERTa-v3-small for Prompt Injection Firewall

This notebook trains `microsoft/deberta-v3-small` on the Prompt Injection Firewall dataset using a free Google Colab GPU (T4).

### Pipeline Overview:
1. Verify GPU acceleration (`torch.cuda.is_available()`)
2. Install required dependencies (`transformers`, `datasets`, `accelerate`, `scikit-learn`)
3. Upload `train.jsonl` and `val.jsonl` (or clone the repository)
4. Fine-tune with mixed precision (`fp16`), evaluate each epoch, save best checkpoint by F1
5. Calibrate decision threshold for target validation False Positive Rate (< 1%)
6. Package and download fine-tuned weights for local deployment

In [ ]:
# 1. Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Select Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# 2. Install dependencies
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
# 3. Upload or Clone Data
# If cloning repo directly:
# !git clone https://github.com/shivanshu-singh-dev/LLM-Prompt-Injection-Scanner.git
# %cd LLM-Prompt-Injection-Scanner

# Or upload train.jsonl and val.jsonl using files.upload():
import os
from google.colab import files

os.makedirs("data/processed", exist_ok=True)
print("Please upload train.jsonl and val.jsonl from your local data/processed directory:")
uploaded = files.upload()
for filename in uploaded.keys():
    os.rename(filename, os.path.join("data/processed", filename))
print("Uploaded files:", os.listdir("data/processed"))

In [ ]:
# 4. Define and Train DeBERTa-v3-small
import json
import numpy as np
from datasets import Dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

def load_split(path):
    texts, labels = [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                d = json.loads(line)
                texts.append(d["text"])
                labels.append(int(d["label"]))
    return Dataset.from_dict({"text": texts, "label": labels})

train_ds = load_split("data/processed/train.jsonl")
val_ds = load_split("data/processed/val.jsonl")
print(f"Loaded {len(train_ds)} train examples, {len(val_ds)} validation examples")

MODEL_ID = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)

def tokenize_fn(batch):
    return tokenizer(batch["text"], max_length=512, truncation=True)

tok_train = train_ds.map(tokenize_fn, batched=True)
tok_val = val_ds.map(tokenize_fn, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1, "precision": p, "recall": r}

training_args = TrainingArguments(
    output_dir="checkpoints",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    gradient_accumulation_steps=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    seed=42,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

OUTPUT_DIR = "models/deberta-v3-firewall"
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

In [ ]:
# 5. Calibrate Decision Threshold for Target FPR on Validation Split
val_preds = trainer.predict(tok_val)
logits = torch.tensor(val_preds.predictions)
probs = torch.softmax(logits, dim=1)[:, 1].numpy()
y_val = np.array(val_ds["label"])

thresholds = np.linspace(0.01, 0.99, 99)
target_fpr = 0.01  # Target: False Positive Rate <= 1%
calibrated_threshold = 0.5

for t in thresholds:
    preds = (probs >= t).astype(int)
    fp = np.sum((y_val == 0) & (preds == 1))
    tn = np.sum((y_val == 0) & (preds == 0))
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    if fpr <= target_fpr:
        calibrated_threshold = t
        break

print(f"Calibrated Threshold for Validation FPR <= {target_fpr*100}%: {calibrated_threshold:.3f}")

with open(os.path.join(OUTPUT_DIR, "threshold.json"), "w") as f:
    json.dump({"calibrated_threshold": float(calibrated_threshold), "target_fpr": target_fpr}, f, indent=2)

In [ ]:
# 6. Package and Download Weights
!zip -r deberta-v3-firewall.zip models/deberta-v3-firewall
files.download("deberta-v3-firewall.zip")
print("Downloaded deberta-v3-firewall.zip! Unzip into models/deberta-v3-firewall in your local workspace.")